# Generateurs de Nombres Aleatoires et Analyse de la Qualite

Ce notebook presente :
1. L'implementation de 7 generateurs (PRNG, CSPRNG, hybride)
2. Une suite de 4 tests statistiques pour evaluer leur qualite
3. Des visualisations comparatives
4. 4 attaques pedagogiques demontrant les faiblesses des PRNG non securises

In [ ]:
import sys
import os
import math
import matplotlib.pyplot as plt
from collections import Counter

sys.path.insert(0, os.path.abspath('.'))

# Generateurs
from GENERATORS.PRNG_non_cryptographics.lcg import lcg, PARAMS_GLIBC
from GENERATORS.PRNG_non_cryptographics.mersenne_twister import generate as mt_generate
from GENERATORS.CSPRNG.bbs import bbs_generate_bytes
from GENERATORS.CSPRNG.hash_drbg import drbg_generate_bytes
from GENERATORS.CSPRNG.os_random import os_generate_bytes
from GENERATORS.Non_deterministic_and_hybrid_generators.xor_nrbg import xor_nrbg
from GENERATORS.PRNG_Gaussian_distribution.box_muller import box_muller_series

# Tests statistiques
from STATISTICS.test_statistique import (
    full_statistical_res, print_res, shannon_entropy_res,
    chi_squared_test, autocorrelation, kolmogorov_smirnov_test
)

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

print("Imports reussis !")

---
## 1. Generation des donnees

On genere 100 000 octets par generateur pour les tests statistiques.

In [ ]:
N_BYTES = 100_000
SEED = 42

print("Generation des donnees pour chaque generateur...\n")

data = {}

# LCG (glibc) - entiers -> octets
print("  LCG (glibc)...", end=" ")
a, c, m = PARAMS_GLIBC['a'], PARAMS_GLIBC['c'], PARAMS_GLIBC['m']
lcg_ints = lcg(SEED, a, c, m, N_BYTES)
data["LCG"] = bytes([x % 256 for x in lcg_ints])
print("OK")

# Mersenne Twister - entiers 32-bit -> octets
print("  Mersenne Twister...", end=" ")
mt_ints = mt_generate(SEED, N_BYTES)
data["Mersenne Twister"] = bytes([x % 256 for x in mt_ints])
print("OK")

# Hash_DRBG (SHA-256) - bytes directement
print("  Hash_DRBG (SHA-256)...", end=" ")
data["Hash_DRBG"] = drbg_generate_bytes(N_BYTES, entropy=b'A'*55, nonce=b'B'*28)
print("OK")

# Blum-Blum-Shub - bytes directement
print("  Blum-Blum-Shub...", end=" ")
data["Blum-Blum-Shub"] = bbs_generate_bytes(N_BYTES, seed=12345)
print("OK")

# os.urandom - bytes directement
print("  os.urandom...", end=" ")
data["os.urandom"] = os_generate_bytes(N_BYTES)
print("OK")

# XOR NRBG (LCG + MT combines via XOR) - entiers -> octets
print("  XOR NRBG...", end=" ")
def lcg_gen(seed, n):
    return lcg(seed, a, c, m, n)
xor_ints = xor_nrbg([lcg_gen, mt_generate], [SEED, SEED + 1], N_BYTES)
data["XOR NRBG"] = bytes([x % 256 for x in xor_ints])
print("OK")

print(f"\nTous les generateurs ont produit {N_BYTES} octets.")

---
## 2. Tests statistiques

On execute 4 tests sur chaque generateur : entropie de Shannon, chi-carre, autocorrelation et Kolmogorov-Smirnov.

In [ ]:
all_results = {}
for name, d in data.items():
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")
    res = full_statistical_res(d)
    print_res(res)
    all_results[name] = res

### 2.1 Tableau comparatif

In [ ]:
print(f"{'Generateur':<25} {'Entropie':>10} {'Chi2':>10} {'AutoCorr':>10} {'KS':>8} {'Score':>7}")
print("-" * 75)

for name, res in all_results.items():
    entropy = res['shannon_entropy']['entropy']
    chi2_st = res['chi_squared']['status']
    ac_pass = all(v['status'] == 'PASS' for v in res['autocorrelation'].values())
    ac_st = "PASS" if ac_pass else "FAIL"
    ks_st = res['kolmogorov_smirnov']['status']

    score = sum([
        res['shannon_entropy']['status'] == 'PASS',
        chi2_st == 'PASS',
        ac_pass,
        ks_st == 'PASS',
    ])

    print(f"{name:<25} {entropy:>10.4f} {chi2_st:>10} {ac_st:>10} {ks_st:>8} {score:>5}/4")

---
## 3. Visualisations

### 3.1 Histogrammes de distribution des octets

Un bon generateur doit produire une distribution uniforme sur [0, 255].

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, (name, d) in enumerate(data.items()):
    ax = axes[idx]
    ax.hist(list(d), bins=256, range=(0, 255), density=True, alpha=0.7, color=f'C{idx}')
    ax.axhline(y=1/256, color='red', linestyle='--', linewidth=1, label='Uniforme theorique')
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.set_xlabel('Valeur octet')
    ax.set_ylabel('Densite')
    ax.legend(fontsize=8)

plt.suptitle('Distribution des octets par generateur', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('histogrammes_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.2 Scatter plots (x_n vs x_{n+1})

Un bon generateur ne doit montrer aucune structure dans le graphe (x_n, x_{n+1}). Le LCG produit des motifs lineaires caracteristiques.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

N_SCATTER = 5000

for idx, (name, d) in enumerate(data.items()):
    ax = axes[idx]
    values = list(d[:N_SCATTER + 1])
    x = values[:-1]
    y = values[1:]
    ax.scatter(x, y, s=0.5, alpha=0.5, color=f'C{idx}')
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.set_xlabel('$x_n$')
    ax.set_ylabel('$x_{n+1}$')
    ax.set_xlim(0, 255)
    ax.set_ylim(0, 255)

plt.suptitle('Scatter plot $x_n$ vs $x_{n+1}$ (correlations sequentielles)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('scatter_plots.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.3 Graphiques d'autocorrelation

In [ ]:
lags = list(range(1, 33))

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, (name, d) in enumerate(data.items()):
    ax = axes[idx]
    corr_values = [autocorrelation(list(d), lag=lag) for lag in lags]

    ax.bar(lags, corr_values, color=f'C{idx}', alpha=0.7)
    threshold = 2 / math.sqrt(len(d))
    ax.axhline(y=threshold, color='red', linestyle='--', linewidth=1, label=f'seuil +/-{threshold:.4f}')
    ax.axhline(y=-threshold, color='red', linestyle='--', linewidth=1)
    ax.axhline(y=0, color='black', linewidth=0.5)
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.set_xlabel('Lag')
    ax.set_ylabel('Autocorrelation')
    ax.legend(fontsize=8)

plt.suptitle('Autocorrelation par lag', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('autocorrelation.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.4 Distribution gaussienne Box-Muller

Verification que la transformee de Box-Muller produit bien une distribution N(0,1).

In [ ]:
# Generateur uniforme [0, 1] base sur MT
def mt_uniform(seed, n):
    ints = mt_generate(seed, n)
    return [x / 0xFFFFFFFF for x in ints]

# Generer 50 000 valeurs gaussiennes via Box-Muller + MT
gaussian_values = box_muller_series(mt_uniform, 42, 50_000)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(gaussian_values, bins=100, density=True, alpha=0.7, color='steelblue', label='Box-Muller')

# Courbe theorique N(0,1)
x_vals = [i * 0.01 for i in range(-400, 401)]
y_vals = [(1 / math.sqrt(2 * math.pi)) * math.exp(-0.5 * x**2) for x in x_vals]
ax.plot(x_vals, y_vals, 'r-', linewidth=2, label='N(0,1) theorique')

ax.set_title('Distribution Box-Muller vs N(0,1)', fontweight='bold')
ax.set_xlabel('Valeur')
ax.set_ylabel('Densite')
ax.legend()
plt.tight_layout()
plt.savefig('box_muller_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Statistiques
mean_val = sum(gaussian_values) / len(gaussian_values)
std_val = math.sqrt(sum((x - mean_val)**2 for x in gaussian_values) / len(gaussian_values))
print(f"Statistiques Box-Muller (50 000 valeurs) :")
print(f"  Moyenne    : {mean_val:.4f} (attendu: 0.0)")
print(f"  Ecart-type : {std_val:.4f} (attendu: 1.0)")
print(f"  Min        : {min(gaussian_values):.4f}")
print(f"  Max        : {max(gaussian_values):.4f}")

### 3.5 Comparaison visuelle de l'entropie

In [ ]:
names = list(all_results.keys())
entropies = [all_results[n]['shannon_entropy']['entropy'] for n in names]
chi2_passed = [all_results[n]['chi_squared']['status'] == 'PASS' for n in names]
colors = ['green' if p else 'red' for p in chi2_passed]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(names, entropies, color=colors, alpha=0.8)
ax.axvline(x=8.0, color='blue', linestyle='--', linewidth=1, label='Maximum theorique (8.0)')
ax.axvline(x=7.9, color='orange', linestyle='--', linewidth=1, label='Seuil acceptable (7.9)')
ax.set_xlabel('Entropie (bits/octet)')
ax.set_title('Entropie de Shannon par generateur', fontweight='bold')
ax.legend()

for bar, val in zip(bars, entropies):
    ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=10)

ax.set_xlim(0, 8.5)
plt.tight_layout()
plt.savefig('entropie_comparaison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Attaques pedagogiques

**AVERTISSEMENT** : Ces attaques sont realisees dans un cadre strictement pedagogique.

### 4.1 Recuperation de la graine LCG

**Modele de menace** : L'attaquant connait les parametres du LCG et observe quelques sorties consecutives. Il retrouve la graine par resolution algebrique, brute-force ou known-plaintext.

In [ ]:
from ATTACKS.lcg_seed_recovery import run_all_attacks as lcg_attacks
lcg_attacks()

### 4.2 Reconstruction d'etat MT19937

**Modele de menace** : L'attaquant observe 624 sorties consecutives du Mersenne Twister et reconstruit l'etat interne complet pour predire toutes les sorties futures.

In [ ]:
from ATTACKS.mt_state_recovery import run_all_attacks as mt_attacks
mt_attacks()

### 4.3 Reutilisation de nonce en AES-CTR

**Modele de menace** : Deux messages sont chiffres avec la meme cle et le meme nonce en mode AES-CTR. L'attaquant recupere le XOR des clairs et utilise le crib-dragging.

In [ ]:
from ATTACKS.aes_ctr_nonce_reuse import demo_aes_ctr_nonce_reuse
demo_aes_ctr_nonce_reuse()

### 4.4 IV previsible en AES-CBC

**Modele de menace** : Le systeme utilise un IV previsible (compteur). L'attaquant peut soumettre des messages choisis et detecter si un message cible a ete chiffre precedemment.

In [ ]:
from ATTACKS.aes_cbc_iv_attack import demo_aes_cbc_iv_attack
demo_aes_cbc_iv_attack()

---
## 5. Tableau comparatif final et recommandations

In [ ]:
print("=" * 85)
print(" TABLEAU COMPARATIF FINAL ".center(85, "="))
print("=" * 85)
print()
print(f"{'Generateur':<25} {'Type':<12} {'Entropie':>8} {'Chi2':>6} {'AC':>6} {'KS':>6} {'Crypto':>8}")
print("-" * 85)

gen_types = {
    "LCG": ("PRNG", "Non"),
    "Mersenne Twister": ("PRNG", "Non"),
    "Hash_DRBG": ("CSPRNG", "Oui"),
    "Blum-Blum-Shub": ("CSPRNG", "Oui"),
    "os.urandom": ("TRNG/OS", "Oui"),
    "XOR NRBG": ("Hybride", "Oui*"),
}

for name, res in all_results.items():
    gtype, crypto = gen_types[name]
    entropy = res['shannon_entropy']['entropy']
    chi2 = res['chi_squared']['status']
    ac = "PASS" if all(v['status'] == 'PASS' for v in res['autocorrelation'].values()) else "FAIL"
    ks = res['kolmogorov_smirnov']['status']
    print(f"{name:<25} {gtype:<12} {entropy:>8.4f} {chi2:>6} {ac:>6} {ks:>6} {crypto:>8}")

print()
print("* XOR NRBG : securite depend de la meilleure source incluse")
print()
print("=" * 85)
print(" RECOMMANDATIONS ".center(85, "="))
print("=" * 85)
print("""
1. Pour la cryptographie : utiliser os.urandom, secrets (Python), ou un CSPRNG
   valide (Hash_DRBG, CTR_DRBG).

2. NE JAMAIS utiliser LCG ou Mersenne Twister pour la generation de cles,
   nonces, IV ou tout usage de securite.

3. Toujours utiliser des nonces/IV uniques et imprevisibles en AES-CTR/CBC.

4. La construction XOR NRBG ameliore la robustesse si au moins une source
   est fiable, mais ne remplace pas un vrai CSPRNG.

5. Blum-Blum-Shub offre une securite theorique forte mais est trop lent
   pour un usage pratique (preferer Hash_DRBG ou os.urandom).
""")